[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/sampling_intervals.ipynb)

# Sampling and confidence intervals full course: the practice set, in Python

Thirty-four exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (several steps), **tier 3** stretch problems (combine ideas, spot the trap, work backwards)
and **tier 4**, the case: the full ledger of 4,000 supplier invoices at Fernmoor Health, a network of fourteen clinics
(a fictional company; our own data), and the director's question: how much were we overcharged last quarter? Each
tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same calculations with Python's
`statistics` module, `numpy`, `pandas` and `scipy.stats`, so you can use them on your own data. And a few cells you
can move yourself: a thousand samples of any size forming their sampling distribution, a hundred confidence intervals
at any level, and a fresh random audit of the ledger.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It runs in your browser, with nothing
to install.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Numbers without
units and without thousands separators: `21875`, not `21,875`. Proportions as decimals: `0.25`, not `25%`. Dollars
without the sign: `59.67`. You can type the arithmetic too: `30/sqrt(25)`. For several answers, separate them with
commas, in the order the question lists them (the box's label shows the order). You can also put your answer between
the quotes in a cell's last line, `practice("P1.4", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[sampling and confidence intervals cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/sampling_intervals_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, random, textwrap
import statistics as st
from statistics import NormalDist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# sampling distributions SKY · standard errors and margins VIOLET · a missed interval CORAL · the one key mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(days?|hours?|minutes?|mins?|invoices?|orders?|customers?|observations?|employees?)\b|[$°]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    if t.endswith("%"):                                   # 25% → 0.25: every proportion here is a decimal
        try: return calc(t[:-1]) / 100
        except Exception: pass
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-6)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Sampling and point estimation',
     "q": "A regional manager will visit 3 of the company's 10 stores, chosen as a simple random sample. How many different samples are possible?",
     "parts": ['number of possible samples'],
     "answer": [120], "tol": [0], "function": 'COMBIN',
     "working": '10 × 9 × 8 / (3 × 2 × 1) = 120'},
    {"id": 'P1.2', "tier": 1, "group": 'Sampling and point estimation',
     "q": 'Six order values from a simple random sample ($): 54, 61, 47, 72, 58, 66. What is the point estimate of the mean order value?',
     "parts": ['point estimate of the mean ($, two decimals)'],
     "answer": [59.67], "tol": [0.01], "function": 'AVERAGE',
     "working": 'sum = 358; 358 / 6 = 59.67'},
    {"id": 'P1.3', "tier": 1, "group": 'Sampling and point estimation',
     "q": 'In a simple random sample of 60 customers, 18 renewed their contract. What is the point estimate of the renewal proportion?',
     "parts": ['point estimate of the proportion (a decimal)'],
     "answer": [0.3], "tol": [0.001], "function": 'COUNTIF, COUNTA',
     "working": 'p̄ = 18 / 60 = 0.30'},
    {"id": 'P1.4', "tier": 1, "group": 'Sampling distributions',
     "q": 'Monthly spend has σ = $30. What is the standard error of the mean of a sample of 25 customers?',
     "parts": ['standard error of x̄ ($)'],
     "answer": [6.0], "tol": [0.01], "function": 'SQRT',
     "working": 'σ/√n = 30 / 5 = 6'},
    {"id": 'P1.5', "tier": 1, "group": 'Sampling distributions',
     "q": '20 % of all orders are returned. What is the standard error of p̄ for samples of 400 orders?',
     "parts": ['standard error of p̄ (a decimal)'],
     "answer": [0.02], "tol": [0.0001], "function": 'SQRT',
     "working": '√(0.2 × 0.8 / 400) = √0.0004 = 0.02'},
    {"id": 'P1.6', "tier": 1, "group": 'Other sampling methods',
     "q": 'A systematic sample of 80 is taken from a list of 2,400 invoices. What is the interval k between selected invoices?',
     "parts": ['interval k'],
     "answer": [30], "tol": [0], "function": '—',
     "working": 'k = N / n = 2400 / 80 = 30'},
    {"id": 'P1.7', "tier": 1, "group": 'Intervals with sigma known',
     "q": 'Which z value leaves 5 % in each tail (a 90 % confidence level)? Give three decimals.',
     "parts": ['z (three decimals)'],
     "answer": [1.645], "tol": [0.0005], "function": 'NORM.S.INV',
     "working": 'NORM.S.INV(0.95) = 1.645'},
    {"id": 'P1.8', "tier": 1, "group": 'Intervals with sigma unknown',
     "q": 'A sample has n = 20. How many degrees of freedom does its t interval use, and what is the t value for 95 % confidence (three decimals)?',
     "parts": ['degrees of freedom', 't value (three decimals)'],
     "answer": [19, 2.093], "tol": [0, 0.0005], "function": 'T.INV.2T',
     "working": 'df = n − 1 = 19; T.INV.2T(0.05, 19) = 2.093'},
    {"id": 'P1.9', "tier": 1, "group": 'Sample size and proportions',
     "q": 'A survey of 300 shoppers finds p̄ = 0.25. What is the estimated standard error of p̄?',
     "parts": ['estimated standard error of p̄ (a decimal)'],
     "answer": [0.025], "tol": [0.0001], "function": 'SQRT',
     "working": '√(0.25 × 0.75 / 300) = 0.025'},
    {"id": 'P1.10', "tier": 1, "group": 'Properties of point estimators',
     "q": 'Four weekly sales figures have squared deviations from their mean that add to 36. What is the unbiased estimate of the population variance?',
     "parts": ['unbiased estimate of the variance, s²'],
     "answer": [12.0], "tol": [0], "function": 'VAR.S',
     "working": 's² = 36/(n − 1) = 36/3 = 12 (dividing by n = 4 gives 9, which runs low on average)'},
    {"id": 'P2.1', "tier": 2, "group": 'Sampling distributions',
     "q": 'Basket values have μ = $120 and σ = $36. For a sample of 81 baskets, what is P(x̄ > $126)?',
     "parts": ['P(x̄ > $126) (four decimals)'],
     "answer": [0.0668], "tol": [0.0005], "function": 'NORM.DIST',
     "working": 'SE = 36/9 = 4; z = (126 − 120)/4 = 1.5; P = 1 − 0.9332 = 0.0668'},
    {"id": 'P2.2', "tier": 2, "group": 'Sampling distributions',
     "q": "35 % of a bank's customers use its app. For samples of 200 customers, what is P(p̄ < 0.30)?",
     "parts": ['P(p̄ < 0.30) (four decimals)'],
     "answer": [0.0691], "tol": [0.0005], "function": 'NORM.S.DIST, SQRT',
     "working": 'np = 70, n(1 − p) = 130 (both ≥ 5); SE = √(0.35 × 0.65/200) = 0.0337; z = −0.05/0.0337 = -1.48; P = 0.0691'},
    {"id": 'P2.3', "tier": 2, "group": 'Sampling distributions',
     "q": 'A plant has 800 machines; maintenance cost per machine has σ = $20. What is the standard error of the mean for a sample of 100 machines (use the finite population correction)?',
     "parts": ['standard error with the correction ($, three decimals)'],
     "answer": [1.872], "tol": [0.001], "function": 'SQRT',
     "working": 'n/N = 12.5 % > 5 %; √(700/799) = 0.9360; SE = 2 × 0.9360 = 1.872'},
    {"id": 'P2.4', "tier": 2, "group": 'Other sampling methods',
     "q": "A distributor's accounts fall in three strata: 500 small (sample mean $40), 300 medium ($65) and 200 large ($120). Estimate the mean across all accounts.",
     "parts": ['stratified estimate of the mean ($)'],
     "answer": [63.5], "tol": [0.01], "function": 'SUMPRODUCT, SUM',
     "working": '0.5 × 40 + 0.3 × 65 + 0.2 × 120 = 20 + 19.5 + 24 = 63.5'},
    {"id": 'P2.5', "tier": 2, "group": 'Intervals with sigma known',
     "q": 'Service calls have σ = 6 min (long records). A sample of 36 calls has x̄ = 48.2 min. Find the 95 % interval for μ.',
     "parts": ['95 % interval (min): lower end', '95 % interval (min): upper end'],
     "answer": [46.24, 50.16], "tol": [0.01, 0.01], "function": 'CONFIDENCE.NORM',
     "working": 'SE = 6/6 = 1; margin = 1.96 × 1 = 1.96; 46.24 to 50.16 min'},
    {"id": 'P2.6', "tier": 2, "group": 'Intervals with sigma unknown',
     "q": 'Eight onboarding calls lasted (min): 23, 27, 21, 30, 25, 28, 24, 26. Find the 95 % interval for the mean (σ unknown).',
     "parts": ['95 % interval (min, two decimals): lower end', '95 % interval (min, two decimals): upper end'],
     "answer": [23.09, 27.91], "tol": [0.01, 0.01], "function": 'AVERAGE, STDEV.S, CONFIDENCE.T',
     "working": 'x̄ = 25.5; s = √(58/7) = 2.8785; t(7) = 2.365; margin = 2.4065; 23.09 to 27.91'},
    {"id": 'P2.7', "tier": 2, "group": 'Sample size and proportions',
     "q": 'Planning value σ = $12. How many customers must be sampled for a 95 % margin of error of $2?',
     "parts": ['sample size n'],
     "answer": [139], "tol": [0], "function": 'ROUNDUP',
     "working": '(1.96 × 12 / 2)² = 11.76² = 138.3 → round up: 139'},
    {"id": 'P2.8', "tier": 2, "group": 'Sample size and proportions',
     "q": '156 of 400 sampled visitors used the new search. Find the 95 % interval for the proportion.',
     "parts": ['95 % interval (a decimal, four places): lower end', '95 % interval (a decimal, four places): upper end'],
     "answer": [0.3422, 0.4378], "tol": [0.0005, 0.0005], "function": 'SQRT, NORM.S.INV',
     "working": 'p̄ = 0.39; SE = √(0.39 × 0.61/400) = 0.0244; margin = 0.0478; 34.2 % to 43.8 %'},
    {"id": 'P2.9', "tier": 2, "group": 'Sample size and proportions',
     "q": 'No earlier estimate exists. How large a sample gives a 95 % margin of error of 0.04 for a proportion?',
     "parts": ['sample size n'],
     "answer": [601], "tol": [0], "function": 'ROUNDUP',
     "working": 'p* = 0.5: 1.96² × 0.25 / 0.0016 = 600.25 → 601'},
    {"id": 'P3.1', "tier": 3, "group": 'Intervals with sigma known',
     "q": 'A report gives a 95 % interval of 71.2 to 78.8 for a mean, with σ = 15 known. What were x̄, the margin of error and n?',
     "parts": ['x̄', 'margin of error E', 'n'],
     "answer": [75.0, 3.8, 60], "tol": [0.01, 0.01, 0], "function": '—',
     "working": 'x̄ = (71.2 + 78.8)/2 = 75; E = 3.8; n = (1.96 × 15/3.8)² = 59.86 ≈ 60'},
    {"id": 'P3.2', "tier": 3, "group": 'Sample size and proportions',
     "q": 'A survey of 100 gave a margin of error of $5. Keeping the confidence level, what sample size gives a margin of $2.50?',
     "parts": ['the new sample size'],
     "answer": [400.0], "tol": [0], "function": '—',
     "working": 'n grows with 1/E²: halving E multiplies n by 4: 400'},
    {"id": 'P3.3', "tier": 3, "group": 'Intervals with sigma unknown',
     "q": "An analyst built a '95 %' interval from n = 12 using z = 1.96 and s. What confidence does that interval really have, and what t value should have been used?",
     "parts": ['its true confidence (a decimal, three places)', 'the t value it should have used (three decimals)'],
     "answer": [0.924, 2.201], "tol": [0.001, 0.0005], "function": 'T.DIST, T.INV.2T',
     "working": 'P(|t₁₁| ≤ 1.96) = T.DIST(1.96, 11, TRUE) × 2 − 1 = 0.924; T.INV.2T(0.05, 11) = 2.201'},
    {"id": 'P3.4', "tier": 3, "group": 'Intervals with sigma known',
     "q": 'With σ known, a 95 % interval has width 10. What is the width of the 99 % interval from the same data?',
     "parts": ['width of the 99 % interval (two decimals)'],
     "answer": [13.14], "tol": [0.01], "function": 'NORM.S.INV',
     "working": 'width ∝ z: 10 × 2.576/1.96 = 13.14'},
    {"id": 'P3.5', "tier": 3, "group": 'Sample size and proportions',
     "q": 'A firm of 1,500 employees wants the share who would use a new benefit within ±5 points (95 %), with no earlier estimate. How many must it survey (use the finite population adjustment)?',
     "parts": ['employees to survey (with the adjustment)'],
     "answer": [307], "tol": [0], "function": 'ROUNDUP',
     "working": 'n₀ = 1.96² × 0.25/0.0025 = 384.2 → 385; n = 385/(1 + 384/1500) = 306.5 → 307'},
    {"id": 'P3.6', "tier": 3, "group": 'Sample size and proportions',
     "q": '90 of 300 sampled stores (from 1,200) missed a delivery window. Find the 95 % interval for the proportion, with the finite population correction.',
     "parts": ['95 % interval (a decimal, four places): lower end', '95 % interval (a decimal, four places): upper end'],
     "answer": [0.2551, 0.3449], "tol": [0.0005, 0.0005], "function": 'SQRT',
     "working": 'p̄ = 0.30; n/N = 25 %; SE = √(0.21/300) × √(900/1199) = 0.0229; margin 0.0449; 25.5 % to 34.5 %'},
    {"id": 'P3.7', "tier": 3, "group": 'Properties of point estimators',
     "q": "Two unbiased estimators of a mean use 50 observations each; their standard errors are 3.0 and 3.6. How many observations would the second need to match the first's standard error?",
     "parts": ['observations the second estimator needs'],
     "answer": [72], "tol": [0], "function": '—',
     "working": 'SE ∝ 1/√n: n = 50 × (3.6/3.0)² = 72'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": 'Reproduce the video: in the random sample of 300 (audit = random or both), what share of invoices had an overcharge?',
     "parts": ['error rate in the random sample (a decimal)'],
     "answer": [0.1], "tol": [0.0005], "function": 'COUNTIFS',
     "working": '30 of 300 = 0.100'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": "Find the 95 % interval for the quarter's error rate, with the finite population correction (N = 4,000).",
     "parts": ['95 % interval for the error rate (a decimal, four places): lower end', '95 % interval for the error rate (a decimal, four places): upper end'],
     "answer": [0.0673, 0.1327], "tol": [0.0005, 0.0005], "function": 'COUNTIFS, SQRT',
     "working": 'SE = √(0.1 × 0.9/300) × √(3700/3999) = 0.0167; margin 0.0327; 6.7 % to 13.3 %'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": "Estimate the quarter's total overcharge with a 95 % interval: x̄ and s of the 300 overcharges, the t margin with the correction, then × 4,000.",
     "parts": ['95 % interval for the total overcharge ($, whole dollars): lower end', '95 % interval for the total overcharge ($, whole dollars): upper end'],
     "answer": [21875, 55415], "tol": [50, 50], "function": 'AVERAGEIFS, STDEV.S, T.INV.2T',
     "working": 'x̄ = 9.66, s = 38.36; SE = s/√300 × 0.9619 = 2.1304; t(299) = 1.968; total = 4000 × 9.66 = 38645 ± 16770'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": "Go further: the shortcut audit of the latest 300 invoices (audit = latest or both). What error rate does it give, and how many of the random sample's standard errors below 10 % does it sit?",
     "parts": ['error rate in the latest 300 (a decimal, four places)', 'standard errors below 10 % (two decimals)'],
     "answer": [0.0433, 3.4], "tol": [0.0005, 0.01], "function": 'COUNTIFS',
     "working": '13/300 = 0.0433; (0.10 − 0.0433)/0.0167 = 3.40; every one of them is from month 3'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": 'Segment: in the random sample, what is the error rate on pharmacy invoices, with its 95 % interval (the ledger has 1,800 pharmacy invoices)?',
     "parts": ['pharmacy error rate (a decimal, four places)', 'its 95 % interval (four places): lower end', 'its 95 % interval (four places): upper end'],
     "answer": [0.1103, 0.0614, 0.1593], "tol": [0.0005, 0.0005, 0.0005], "function": 'COUNTIFS, SQRT',
     "working": '16 of 145 = 0.1103; margin 0.0489 (with √((1800 − 145)/1799))'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": 'Sensitivity: the 99 % interval for the total overcharge. Is its lower end still above the $12,000 a quarter that an automatic invoice check would cost?',
     "parts": ['99 % interval for the total overcharge ($, whole dollars): lower end', '99 % interval for the total overcharge ($, whole dollars): upper end'],
     "answer": [16553, 60736], "tol": [50, 50], "function": 'T.INV.2T',
     "working": 't(299) for 99 % = 2.592; margin 4000 × 2.592 × 2.1304 = 22091; lower end 16553 > 12000: yes'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": "Plan the next audit: how many invoices give the error rate within ±2 points at 95 %, using this audit's 10 % as p*, with the finite population adjustment (N = 4,000)?",
     "parts": ['invoices to audit next time'],
     "answer": [712], "tol": [0], "function": 'ROUNDUP',
     "working": 'n₀ = 1.96² × 0.1 × 0.9/0.0004 = 864.4 → 865; n = 865/(1 + 864/4000) = 711.3 → 712'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study: Fernmoor Health',
     "q": "Write the recommendation to the finance director in two sentences, with one number that anchors it. (The check cell takes the anchor: the 95 % lower bound on the quarter's overcharges, to the nearest $100.)",
     "parts": ['the anchor: the 95 % lower bound on the overcharges ($, to the nearest 100)'],
     "answer": [21900.0], "tol": [100], "function": '—',
     "working": "'We were overcharged about $38,600 last quarter, and at least $21,875 with 95 % confidence, mostly by the pharmacy supplier before month 3. Claim it back, and install the automatic check ($12,000 a quarter).'"},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 34 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 9, 7, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def close(a, b, tol=1e-9):
    return all(abs(x - y) <= tol for x, y in zip(a, b)) and len(a) == len(b)

## Tier 1 · Warm-up

One idea, one step, small numbers: how many samples are possible, a point estimate of a mean and of a proportion,
the standard error of x̄ and of p̄, a systematic sample's interval, a z value, degrees of freedom and a t value, and
the variance with n − 1.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

`math.comb(N, n)` counts the possible samples. `statistics.mean` gives a point estimate. A standard error is one line
of arithmetic. For z values, `statistics.NormalDist().inv_cdf(…)` (or `scipy.stats.norm.ppf`) is the spreadsheet's
NORM.S.INV; for t values, `scipy.stats.t.ppf(1 − α/2, df)` is T.INV.2T(α, df). `statistics.variance` divides by
n − 1. Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
print("possible samples:", math.comb(10, 3))
orders6 = [54, 61, 47, 72, 58, 66]                                 # P1.2
print("x̄:", round(st.mean(orders6), 2), "   p̄:", 18 / 60, "   SE of x̄:", 30 / math.sqrt(25),
      "   SE of p̄:", round(math.sqrt(0.2 * 0.8 / 400), 4))
z90 = NormalDist().inv_cdf(0.95)
t19 = stats.t.ppf(0.975, 19)
print("z for 90 %:", round(z90, 3), " (scipy:", round(stats.norm.ppf(0.95), 3), ")   t, 19 df, 95 %:", round(t19, 3))
weeks = [9, 15, 9, 15]                                              # four values whose squared deviations add to 36
print("variance with n − 1:", st.variance(weeks), "   with n:", st.pvariance(weeks))

assert [math.comb(10, 3)] == answers("P1.1") and [round(st.mean(orders6), 2)] == answers("P1.2")
assert [18 / 60] == answers("P1.3") and [30 / math.sqrt(25)] == answers("P1.4")
assert close([math.sqrt(0.2 * 0.8 / 400)], answers("P1.5")) and [2400 // 80] == answers("P1.6")
assert [round(z90, 3)] == answers("P1.7") and [19, round(t19, 3)] == answers("P1.8")
assert close([math.sqrt(0.25 * 0.75 / 300)], answers("P1.9")) and [st.variance(weeks)] == answers("P1.10")

### The population, and one simple random sample

The course's population: an online store's 5,000 orders last month (order value in $, right-skewed, and whether it
went express). Only a simulation lets us see a whole population, so this one is built from a seeded generator:
everyone gets exactly the same 5,000 orders, the same as the spreadsheet's Try it tab. Then one simple random sample of
30, drawn without replacement: the course's x̄ = $81.75, s = $48.24 and p̄ = 7/30.

In [ ]:
def build_orders():
    r = random.Random(703)
    raw = [math.exp(r.gauss(4.2, 0.6)) for _ in range(5000)]
    m = math.fsum(raw) / len(raw); s = math.sqrt(math.fsum((x - m) ** 2 for x in raw) / len(raw))
    vals = [round(80 + 50 * (x - m) / s, 2) for x in raw]
    express = set(r.sample(range(5000), 1500))
    return pd.DataFrame({"order": range(1, 5001), "value": vals,
                         "express": [1 if i in express else 0 for i in range(5000)]})

orders = build_orders()
mu, sigma, p = orders["value"].mean(), orders["value"].std(ddof=0), orders["express"].mean()
sample = orders.iloc[random.Random(51).sample(range(5000), 30)]          # 30 row positions, without replacement
xbar, s, pbar = sample["value"].mean(), sample["value"].std(), sample["express"].mean()
print(f"population: μ = ${mu:.2f}, σ = ${sigma:.2f}, p = {p:.2f}")
print(f"sample of 30: x̄ = ${xbar:.2f}, s = ${s:.2f}, p̄ = {sample['express'].sum()}/30 = {pbar:.4f}")
print(f"sampling error this time: x̄ − μ = ${xbar - mu:.2f}")
assert (round(mu, 2), round(sigma, 2), p) == (80.0, 50.0, 0.3)
assert (round(xbar, 2), round(s, 2), sample["express"].sum()) == (81.75, 48.24, 7)

fig, ax = plt.subplots(figsize=(11.5, 4))
fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.16)
ax.hist(orders["value"], bins=np.arange(0, orders["value"].max() + 20, 20), color=GHOST, edgecolor=SLATE)
ax.scatter(sample["value"], np.full(30, 8), s=60, color=CHALK, zorder=3)
ax.axvline(mu, color=SKY, lw=2.4); ax.axvline(xbar, color=GOLD, lw=2.4, ls="--")
ax.set_xlabel(r"order value (\$)"); ax.set_ylabel("orders")
fig.suptitle(rf"5,000 orders (μ = \${mu:.2f}, sky) and one sample of 30 (x̄ = \${xbar:.2f}, gold)", x=0.07, ha="left")
footnote(fig, "a simulation: the whole population known · computed"); plt.show()

## Tier 2 · Core

The course's standard problems, several steps each: a probability for x̄ and for p̄, the finite population
correction, a stratified estimate, intervals with σ known and unknown, the sample size for a mean, the interval for a
proportion, and the sample size for a proportion.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

### How you'd do this in Python

**Probabilities.** `scipy.stats.norm.cdf(x, loc=μ, scale=SE)` is NORM.DIST(x, μ, SE, TRUE); `norm.sf` is one minus it.

**Intervals.** With σ known, `scipy.stats.norm.interval(0.95, loc=x̄, scale=σ/√n)`; with σ unknown,
`scipy.stats.t.interval(0.95, n − 1, loc=x̄, scale=s/√n)`. Both return the lower and upper ends. `scipy.stats.sem`
gives s/√n straight from the data.

**Sample sizes.** Plain arithmetic, rounded **up** with `math.ceil`: rounding down would miss the margin you asked for.

In [ ]:
print("P(x̄ > 126):", round(stats.norm.sf(126, loc=120, scale=36 / math.sqrt(81)), 4))
se22 = math.sqrt(0.35 * 0.65 / 200)
print("P(p̄ < 0.30):", round(stats.norm.cdf(0.30, loc=0.35, scale=se22), 4))
fpc = math.sqrt((800 - 100) / (800 - 1))
print("SE with the correction:", round(20 / math.sqrt(100) * fpc, 3))
print("stratified mean:", np.average([40, 65, 120], weights=[500, 300, 200]))
assert [round(stats.norm.sf(126, 120, 4), 4)] == answers("P2.1") and [round(stats.norm.cdf(0.30, 0.35, se22), 4)] == answers("P2.2")
assert [round(2 * fpc, 3)] == answers("P2.3") and [np.average([40, 65, 120], weights=[500, 300, 200])] == answers("P2.4")

lo, hi = stats.norm.interval(0.95, loc=48.2, scale=6 / math.sqrt(36))      # σ known
print("σ known:", round(lo, 2), "to", round(hi, 2))
calls = [23, 27, 21, 30, 25, 28, 24, 26]                                     # σ unknown
lo_t, hi_t = stats.t.interval(0.95, len(calls) - 1, loc=np.mean(calls), scale=stats.sem(calls))
print("σ unknown:", round(lo_t, 2), "to", round(hi_t, 2), "  (t =", round(stats.t.ppf(0.975, 7), 3), ")")
assert [round(lo, 2), round(hi, 2)] == answers("P2.5") and [round(lo_t, 2), round(hi_t, 2)] == answers("P2.6")

pb = 156 / 400
lo_p, hi_p = pb - 1.96 * math.sqrt(pb * (1 - pb) / 400), pb + 1.96 * math.sqrt(pb * (1 - pb) / 400)
print("proportion:", round(lo_p, 4), "to", round(hi_p, 4))
print("n for a mean:", math.ceil((1.96 * 12 / 2) ** 2), "   n for a proportion:", math.ceil(1.96 ** 2 * 0.25 / 0.04 ** 2))
assert [round(lo_p, 4), round(hi_p, 4)] == answers("P2.8")
assert [math.ceil((1.96 * 12 / 2) ** 2)] == answers("P2.7") and [math.ceil(1.96 ** 2 * 0.25 / 0.04 ** 2)] == answers("P2.9")

### The sampling distribution, by simulation

Draw a sample at random, take its mean, and repeat a thousand times. The means pile up into the sampling
distribution of x̄: centred on μ, with a spread close to σ/√n (the standard error), and closer to a bell than the
population is. The slider sets the sample size. At n = 2 the pile is still skewed like the orders; at n = 30 it's
nearly normal, and four times the sample size halves its spread.

In [ ]:
# the whole simulation, in five lines
rng = random.Random(1006)
means = []
for _ in range(1000):
    means.append(st.fmean(rng.sample(list(orders["value"]), 30)))      # draw 30 at random, take the mean, repeat
plt.hist(means, bins=30, color=SKY, edgecolor=SLATE); plt.xlabel(r"sample mean (\$)"); plt.show()
assert (round(st.fmean(means), 2), round(st.stdev(means), 2)) == (79.72, 9.05)            # the course's simulation

In [ ]:
VALUES = orders["value"].to_numpy()

def sample_means(n, reps=1000, seed=1006):
    r = random.Random(seed)
    return np.array([VALUES[r.sample(range(5000), n)].mean() for _ in range(reps)])

def sampling_distribution(n=30):
    means = sample_means(n)
    se = sigma / math.sqrt(n) * math.sqrt((5000 - n) / (5000 - 1))
    fig, ax = plt.subplots(figsize=(11.5, 4.4))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.15)
    ax.hist(means, bins=np.arange(0, 260, 2.5), color=SKY, edgecolor=SLATE)
    xs = np.linspace(0, 260, 600)
    ax.plot(xs, 1000 * 2.5 * stats.norm.pdf(xs, mu, se), color=GOLD, lw=2)
    ax.axvline(mu, color=CHALK, lw=1.4, ls=":")
    ax.set_xlim(0, 260); ax.set_xlabel(r"sample mean x̄ (\$)"); ax.set_ylabel("samples")
    fig.suptitle(f"1,000 samples of {n}: mean of the means {means.mean():.2f}, their s.d. {means.std(ddof=1):.2f}, "
                 f"σ/√n × correction {se:.2f}, skewness {stats.skew(means, bias=False):.1f}", x=0.07, ha="left", fontsize=13)
    footnote(fig, "simulated · 1,000 samples · seeded"); plt.show()

m30 = sample_means(30)
assert (round(m30.mean(), 2), round(m30.std(ddof=1), 2)) == (79.72, 9.05)            # the course's simulation
assert round(sigma / math.sqrt(30), 2) == 9.13

In [ ]:
interact(sampling_distribution, n=IntSlider(value=30, min=2, max=120, step=1, description="sample size"));

## Tier 3 · Stretch

Combine two ideas, spot the trap, work backwards: an interval taken apart, how the margin scales with n, z used where
t belonged, a wider level, a survey of a small firm, a proportion with the correction, and the price of an
inefficient estimator.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

### How you'd do this in Python

Working backwards is algebra on the interval: x̄ is its midpoint, E half its width. The z-with-s trap is a t
probability: `scipy.stats.t.cdf(1.96, df)` says how much of the t curve lies inside ± 1.96. A finite population
changes both the standard error and the sample size; write each as a small function and reuse it.

In [ ]:
lo31, hi31 = 71.2, 78.8
x31, E31 = (lo31 + hi31) / 2, (hi31 - lo31) / 2
print("x̄:", x31, "  E:", round(E31, 2), "  n:", round((1.96 * 15 / E31) ** 2, 2))
cover = 2 * stats.t.cdf(1.96, 11) - 1
print("a '95 %' z interval with s and n = 12 really covers", round(cover, 3), "  t should be", round(stats.t.ppf(0.975, 11), 3))
assert close([x31, E31, round((1.96 * 15 / E31) ** 2)], answers("P3.1"))
assert [round(cover, 3), round(stats.t.ppf(0.975, 11), 3)] == answers("P3.3")

def n_proportion(E, p_star=0.5, z=1.96, N=None):
    n0 = math.ceil(z ** 2 * p_star * (1 - p_star) / E ** 2)
    return n0 if N is None else math.ceil(n0 / (1 + (n0 - 1) / N))

def se_proportion(pbar, n, N=None):
    se = math.sqrt(pbar * (1 - pbar) / n)
    return se if N is None else se * math.sqrt((N - n) / (N - 1))

print("survey of a firm of 1,500:", n_proportion(0.05), "→", n_proportion(0.05, N=1500))
e36 = 1.96 * se_proportion(0.3, 300, N=1200)
print("missed windows:", round(0.3 - e36, 4), "to", round(0.3 + e36, 4))
assert [n_proportion(0.05, N=1500)] == answers("P3.5") and [round(0.3 - e36, 4), round(0.3 + e36, 4)] == answers("P3.6")
assert [100 * (5 / 2.5) ** 2] == answers("P3.2") and [round(10 * 2.576 / 1.96, 2)] == answers("P3.4")
assert [round(50 * (3.6 / 3.0) ** 2)] == answers("P3.7")

### What 95 % confidence means

The course's warehouse: pick times with a true mean of 12 minutes and σ = 1.5 minutes. Draw a hundred samples of 36,
and build each one's interval, x̄ ± z σ/√n. At 95 %, 94 of the course's hundred cover the true mean and 6 miss (in
coral). The confidence belongs to the method: about 95 in every 100 such intervals cover μ. Move the level and watch
the intervals widen and the misses become rarer.

In [ ]:
def intervals(level=95, n=36, seed=95):
    """A hundred samples of n from the process (μ = 12, σ = 1.5), each with its interval x̄ ± z σ/√n."""
    z = round(NormalDist().inv_cdf(0.5 + level / 200), 3)
    r = random.Random(seed)
    e = z * 1.5 / math.sqrt(n)
    bars = []
    for _ in range(100):
        xb = st.fmean([r.gauss(12, 1.5) for _ in range(n)])
        bars.append((xb - e, xb + e))
    return bars, e

def hundred_intervals(level=95, n=36):
    bars, e = intervals(level, n)
    hits = sum(lo <= 12 <= hi for lo, hi in bars)
    fig, ax = plt.subplots(figsize=(11.5, 6))
    fig.subplots_adjust(left=0.05, right=0.98, top=0.9, bottom=0.1)
    for k, (lo, hi) in enumerate(bars):
        ax.plot([lo, hi], [k, k], color=VIOLET if lo <= 12 <= hi else CORAL, lw=2.2 if lo <= 12 <= hi else 3)
    ax.axvline(12, color=GOLD, lw=2.4)
    ax.set_xlim(10.4, 13.6); ax.set_ylim(-1, 100); ax.set_yticks([]); ax.set_xlabel("minutes")
    fig.suptitle(f"{level} % intervals, samples of {n} (margin ± {e:.2f}): {hits} of 100 cover μ = 12", x=0.05, ha="left")
    footnote(fig, "simulated · 100 samples · seeded"); plt.show()

assert round(intervals()[1], 2) == 0.49 and sum(lo <= 12 <= hi for lo, hi in intervals()[0]) == 94   # the course's 94

In [ ]:
interact(hundred_intervals, level=IntSlider(value=95, min=50, max=99, step=1, description="level (%)"),
         n=IntSlider(value=36, min=4, max=144, step=4, description="sample size"));

## Tier 4 · The case: Fernmoor Health's overcharges

Fernmoor Health runs fourteen clinics (a fictional company; our own data). Its finance director asks: **how much
were we overcharged by our suppliers last quarter, in total?** The quarter's ledger has 4,000 supplier invoices, in
date order. Auditing an invoice (checking it against the contract price) takes time, so the team audits a sample.
First they took a shortcut: the latest 300 invoices. Then the director asked for a simple random sample of 300 from
the whole quarter.

| column | what it is |
|---|---|
| `invoice` | invoice number, in date order (1–4,000) |
| `month` | month of the quarter: 1, 2 or 3 |
| `supplier` | pharmacy, supplies or equipment |
| `billed` | amount billed, $ |
| `random` | the random number that drew the audit (made once, kept): the 300 smallest are the random sample |
| `audit` | random (in the simple random sample), latest (in the latest 300), both, or blank (not audited) |
| `overcharge` | billed minus the correct amount, $ (0 when correct; blank when not audited) |

The next cell builds the ledger from seeded generators, so everyone gets exactly the same 4,000 invoices (the same as
the spreadsheet's Case study tab), and checks it against the course's case numbers.

In [ ]:
def build_ledger():
    r = random.Random(2027)
    types = ["pharmacy"] * 1800 + ["supplies"] * 1600 + ["equipment"] * 600
    r.shuffle(types); rows = []
    for i, t in enumerate(types):
        month = 1 + i * 3 // 4000
        billed = round(math.exp(r.gauss({"pharmacy": 6.6, "supplies": 6.2, "equipment": 7.9}[t], 0.55)), 2)
        pe = {"pharmacy": 0.18 if month < 3 else 0.02, "supplies": 0.08, "equipment": 0.06}[t]
        err = r.random() < pe
        over = round(billed * r.uniform(0.05, 0.15), 2) if err else 0.0
        rows.append((i + 1, month, t, billed, over))
    return rows

def rand_column(n, chosen, seed):
    """n different random numbers (six places) whose len(chosen) smallest sit on the chosen rows: RAND, sorted."""
    r = random.Random(seed)
    u = sorted({round(r.random(), 6) for _ in range(n + 50)})
    u = sorted(r.sample(u, n))
    first, rest = list(chosen), [i for i in range(n) if i not in set(chosen)]
    r.shuffle(first); r.shuffle(rest)
    out = [0.0] * n
    for k, i in enumerate(first + rest): out[i] = u[k]
    return out

LEDGER = build_ledger()
N_INV, N_AUDIT = len(LEDGER), 300
RANDOM_IDX = sorted(random.Random(2028).sample(range(N_INV), N_AUDIT))
LATEST_IDX = list(range(N_INV - N_AUDIT, N_INV))
rand_set, late_set = set(RANDOM_IDX), set(LATEST_IDX)
ledger = pd.DataFrame(LEDGER, columns=["invoice", "month", "supplier", "billed", "true_overcharge"])
ledger.insert(4, "random", rand_column(N_INV, RANDOM_IDX, 2029))
ledger["audit"] = ["both" if i in rand_set and i in late_set else "random" if i in rand_set else
                   "latest" if i in late_set else "" for i in range(N_INV)]
ledger["overcharge"] = ledger["true_overcharge"].where(ledger["audit"] != "")     # blank (NaN) when not audited
TRUTH = ledger.pop("true_overcharge")                                           # known only because we built it

# The case's numbers as the course computed them, to check this ledger against.
# @STATS
CASE_STATS = {'e': 30,
 'e_latest': 13,
 'p': 0.1,
 'p_latest': 0.0433333333,
 'fpc': 0.9618894468,
 'se_p': 0.0166604139,
 'E_p': 0.0326544113,
 'xbar': 9.6611666667,
 's': 38.3621014684,
 't': 1.9679296691,
 'se_x': 2.1304282991,
 'E_x': 4.1925330576,
 'total': 38644.6666666667,
 'E_total': 16770.1322302326,
 't99': 2.5923718841,
 'E_total99': 22091.4496945725,
 'by_month': {1: (11, 83), 2: (13, 106), 3: (6, 111)},
 'latest_months': [3],
 'billed_pop': 1099.438125,
 'billed_samp': 1068.8526666667,
 'billed_E': 130.6744138755,
 'cost': 12000,
 'nonzero': 30,
 'n_next0': 865,
 'n_next': 712,
 'pharm_n': 145,
 'pharm_e': 16,
 'pharm_p': 0.1103448276,
 'pharm_E': 0.0489150309,
 'true_total': 39061.94,
 'true_p': 0.10075}
# @END STATS
S = CASE_STATS
rs = ledger[ledger["audit"].isin(["random", "both"])]                          # the random sample
ls = ledger[ledger["audit"].isin(["latest", "both"])]                          # the shortcut
assert len(ledger) == 4000 and len(rs) == len(ls) == 300
assert ledger.nsmallest(300, "random")["invoice"].tolist() == sorted(rs["invoice"], key=lambda k: ledger["random"][k - 1])
assert (rs["overcharge"] > 0).sum() == S["e"] and (ls["overcharge"] > 0).sum() == S["e_latest"]
assert abs(rs["overcharge"].mean() - S["xbar"]) < 1e-9 and abs(rs["overcharge"].std() - S["s"]) < 1e-9
assert ledger["supplier"].value_counts().to_dict() == {"pharmacy": 1800, "supplies": 1600, "equipment": 600}
assert sorted(ls["month"].unique()) == S["latest_months"] and abs(TRUTH.sum() - S["true_total"]) < 1e-6
print(f"{len(ledger)} invoices, {ledger.shape[1]} columns: the same data as the course's case.")
ledger.head(8)

### How you'd do this in Python

With the ledger in a DataFrame, each step of the case is a line or two: `isin` picks the audited rows; the error rate
is the mean of `overcharge > 0`; `scipy.stats.t.ppf` gives t with 299 degrees of freedom; the finite population
correction is one factor, because 300 is 7.5 % of the ledger; and the total is 4,000 times the mean. `groupby("month")`
shows why the shortcut misled. These cells reproduce the video's case numbers. Then it's your turn.

In [ ]:
n, N = len(rs), len(ledger)
fpc_case = math.sqrt((N - n) / (N - 1))
p_hat = (rs["overcharge"] > 0).mean()
se_p = math.sqrt(p_hat * (1 - p_hat) / n) * fpc_case
print(f"error rate: random sample {p_hat:.3f} ± {1.96 * se_p:.4f} (95 %), shortcut {(ls['overcharge'] > 0).mean():.3f}")
print("errors by month in the random sample:\n",
      rs.assign(wrong=rs["overcharge"] > 0).groupby("month")["wrong"].agg(["sum", "count"]).to_string())
x_hat, s_hat = rs["overcharge"].mean(), rs["overcharge"].std()
t299 = stats.t.ppf(0.975, n - 1)
se_x = s_hat / math.sqrt(n) * fpc_case
print(f"\nmean overcharge per invoice: ${x_hat:.2f} (s = ${s_hat:.2f}); t = {t299:.3f}; SE = ${se_x:.2f}")
print(f"total: ${N * x_hat:,.0f} ± ${N * t299 * se_x:,.0f}  →  ${N * (x_hat - t299 * se_x):,.0f} to ${N * (x_hat + t299 * se_x):,.0f}")
assert round(p_hat, 3) == 0.1 and round(se_p, 4) == 0.0167 and round(t299, 3) == 1.968
assert round(N * x_hat, -2) == 38600 and round(N * t299 * se_x, -2) == 16800

In [ ]:
by_month = rs.assign(wrong=rs["overcharge"] > 0).groupby("month")["wrong"].mean()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1, 1.3]})
fig.subplots_adjust(left=0.07, right=0.98, top=0.8, bottom=0.15, wspace=0.3)
ax1.bar([1, 2, 3], 100 * by_month, color=[CHALK, CHALK, SKY], width=0.6)
ax1.set_xticks([1, 2, 3], ["month 1", "month 2", "month 3"]); ax1.set_ylabel("invoices with an overcharge (%)")
ax1.set_title("the shortcut audited month 3 only", loc="left", fontsize=13)
lo_t, hi_t = N * (x_hat - t299 * se_x), N * (x_hat + t299 * se_x)
t99 = stats.t.ppf(0.995, n - 1)
ax2.plot([lo_t, hi_t], [1, 1], color=VIOLET, lw=6, solid_capstyle="butt"); ax2.scatter([N * x_hat], [1], color=GOLD, s=90, zorder=3)
ax2.plot([N * (x_hat - t99 * se_x), N * (x_hat + t99 * se_x)], [0.5, 0.5], color=VIOLET, lw=3, alpha=0.6)
ax2.axvline(12000, color=CORAL, ls=":", lw=2); ax2.text(12000, 1.35, " the automatic check: $12,000", color=CORAL, fontsize=11)
ax2.set_yticks([0.5, 1], ["99 %", "95 %"]); ax2.set_ylim(0, 1.6); ax2.set_xlim(0, 70000)
ax2.set_xlabel(r"total overcharge last quarter (\$)")
ax2.set_title("the total, with its intervals", loc="left", fontsize=13)
fig.suptitle("Fernmoor Health · 4,000 invoices, a random audit of 300", x=0.07, ha="left", fontsize=15)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: the shortcut measured in standard errors, the pharmacy invoices
on their own, the 99 % interval, the next audit's size, and the recommendation with its number. Use the DataFrame
`ledger` in a cell of your own (press **+** to add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### Another random audit

The audit could have drawn any 300 invoices. The slider draws a different simple random sample for each seed and
builds its 95 % interval for the total. Because we built this ledger, we know the true total (gold): watch how often
the interval covers it. About 95 seeds in a hundred should, which is what the confidence means.

In [ ]:
def another_audit(seed=2028):
    idx = random.Random(seed).sample(range(N), 300)
    ov = TRUTH.iloc[idx]
    xb, sd = ov.mean(), ov.std()
    e = N * stats.t.ppf(0.975, 299) * sd / math.sqrt(300) * fpc_case
    fig, ax = plt.subplots(figsize=(11.5, 2.8))
    fig.subplots_adjust(left=0.05, right=0.98, top=0.72, bottom=0.25)
    hit = N * xb - e <= TRUTH.sum() <= N * xb + e
    ax.plot([N * xb - e, N * xb + e], [0, 0], color=VIOLET if hit else CORAL, lw=8, solid_capstyle="butt")
    ax.scatter([N * xb], [0], color=CHALK, s=80, zorder=3)
    ax.axvline(TRUTH.sum(), color=GOLD, lw=2.4)
    ax.set_xlim(0, 80000); ax.set_yticks([]); ax.set_xlabel(r"total overcharge (\$)")
    fig.suptitle(rf"seed {seed}: \${N * xb:,.0f} ± \${e:,.0f}; the true total \${TRUTH.sum():,.0f} is "
                 + ("inside" if hit else "outside"), x=0.05, ha="left", fontsize=14)
    footnote(fig, "a fictional company · our own data · computed"); plt.show()

hits = 0
for sd_ in range(100):
    ov_ = TRUTH.iloc[random.Random(sd_).sample(range(N), 300)]
    e_ = stats.t.ppf(0.975, 299) * ov_.std() / math.sqrt(300) * fpc_case
    hits += abs(ov_.mean() - TRUTH.mean()) <= e_
print(f"{hits} of 100 random audits (seeds 0–99) cover the true total.")
assert 85 <= hits <= 100

In [ ]:
interact(another_audit, seed=IntSlider(value=2028, min=2000, max=2100, step=1, description="seed"));

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `ledger` with pandas and scipy and checks it
against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
def wrong(d): return (d["overcharge"] > 0).mean()
def total_interval(d, level):
    t = stats.t.ppf(0.5 + level / 2, len(d) - 1)
    e = N * t * d["overcharge"].std() / math.sqrt(len(d)) * math.sqrt((N - len(d)) / (N - 1))
    return N * d["overcharge"].mean() - e, N * d["overcharge"].mean() + e

mine = {}
mine["P4.1"] = [wrong(rs)]
mine["P4.2"] = [round(wrong(rs) - 1.96 * se_p, 4), round(wrong(rs) + 1.96 * se_p, 4)]
mine["P4.3"] = [round(v) for v in total_interval(rs, 0.95)]
mine["P4.4"] = [round(wrong(ls), 4), round((wrong(rs) - wrong(ls)) / se_p, 2)]
ph = rs[rs["supplier"] == "pharmacy"]
n_ph, N_ph = len(ph), (ledger["supplier"] == "pharmacy").sum()
e_ph = 1.96 * se_proportion(wrong(ph), n_ph, N=N_ph)
mine["P4.5"] = [round(wrong(ph), 4), round(wrong(ph) - e_ph, 4), round(wrong(ph) + e_ph, 4)]
mine["P4.6"] = [round(v) for v in total_interval(rs, 0.99)]
mine["P4.7"] = [n_proportion(0.02, p_star=round(wrong(rs), 2), N=N)]
mine["P4.8"] = [round(total_interval(rs, 0.95)[0], -2)]

for k, got in mine.items():
    assert len(got) == len(answers(k)) and all(right(float(g), w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"])), (k, got)
    print(f"{k}: " + " · ".join(f"{p}: {show(float(g))}" for p, g in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")

## Sources
Every exercise, data set and company here is our own (Fernmoor Health and its ledger are fictional). The chapter order
follows a standard first course in business statistics (sampling and sampling distributions, interval estimation, and
the useful parts of sample surveys); the definitions agree with OpenStax, Introductory Business Statistics 2e
(retrieved 2026-11-07).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs03_sampling_intervals/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs03_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert close(_q["answer"], [float(v) for v in _flat(_e["answer"])]), _e["id"]
        assert _q["tol"] == (list(_e["tol"]) if isinstance(_e["tol"], tuple) else [_e["tol"]] * len(_q["answer"])), _e["id"]
    assert len(_P.EX) == len(EX)
    assert _P.CASE["billed"] == ledger["billed"].tolist() and _P.CASE["audit"] == ledger["audit"].tolist()
    assert [r_[4] for r_ in _P.LEDGER] == TRUTH.tolist()
    print("matches the course's practice set")